# 🍜 STEP 01 — Semantic Kernel 기초: Kernel & Plugin

**F&B 창업 지원 멀티 에이전트 시스템** 학습 시리즈 1단계입니다.

---

## 이 노트북에서 배울 것

| 개념 | 설명 | F&B 시스템 연결 |
|------|------|----------------|
| `Kernel` | SK의 핵심 엔진. AI 서비스 + 플러그인을 연결 | 모든 에이전트의 베이스 |
| `@kernel_function` | Python 함수를 LLM이 호출 가능한 툴로 등록 | 카카오 API, 법령 API 툴 구현 방식 |
| `KernelPlugin` | 관련 함수들의 묶음 | LegalTaxPlugin, LocationPlugin 등 |
| `FunctionChoiceBehavior.Auto()` | LLM이 알아서 함수를 선택·호출 | 에이전트 자율 판단 |

---

## 전체 학습 로드맵
```
STEP 01  →  STEP 02  →  STEP 03  →  STEP 04  →  STEP 05
Kernel      단일 에이전트   멀티 에이전트   MCP 연동     HITL + Sign-off
Plugin      ChatAgent     GroupChat     API Tool     Process Framework
```

## 0. 설치 & 환경 설정

In [1]:
# semantic-kernel 최신 버전 설치
# [mcp] 옵션은 STEP 04에서 사용
%pip install --upgrade semantic-kernel
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# .env 파일을 이 노트북과 같은 폴더에 만들어 두세요
# .env 내용 예시:
#   AZURE_OPENAI_ENDPOINT=https://xxxxxx.openai.azure.com/
#   AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=gpt-4o
#   AZURE_OPENAI_API_KEY=xxxxxxxxxxxx
#   AZURE_OPENAI_API_VERSION=2025-03-01-preview

import os
from dotenv import load_dotenv
load_dotenv()

AZURE_OPENAI_ENDPOINT          = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT        = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")
AZURE_OPENAI_API_KEY           = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION       = os.getenv("AZURE_OPENAI_API_VERSION", "2025-03-01-preview")

print("환경변수 로드 완료:", AZURE_OPENAI_ENDPOINT)

환경변수 로드 완료: https://student02-11-1604-resource.cognitiveservices.azure.com/


---
## 1. Kernel 만들기

```
Kernel
 ├── AI Service (Azure OpenAI / OpenAI / Ollama ...)
 └── Plugins (함수 묶음들)
```

> **핵심 개념**: Kernel은 AI 서비스와 도구(플러그인)를 연결하는 중앙 허브입니다.

In [3]:
import asyncio
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion


def create_Kernel() -> Kernel:
    """Kernel 인스턴스를 생성하고 Azure OpenAi 서비스를 등록합니다."""
    kernel = Kernel()
    kernel.add_service(
        AzureChatCompletion(
            deployment_name=AZURE_OPENAI_DEPLOYMENT,
            endpoint=AZURE_OPENAI_ENDPOINT,
            api_key=AZURE_OPENAI_API_KEY,
            api_version=AZURE_OPENAI_API_VERSION,
        )
    )
    return kernel


kernel = create_Kernel()
print("Kernel 생성 완료! 등록된 서비스:", list(kernel.services.keys()))

Kernel 생성 완료! 등록된 서비스: ['gpt-4o']


---
## 2. 가장 단순한 사용 — 프롬프트 직접 실행

> `kernel.invoke_prompt()` 로 LLM을 바로 호출할 수 있습니다.

In [4]:
# async def test_basic_invoke():
#     result = await kernel.invoke_prompt(
#         "서울 마포구에서 일반음식점을 창업할 때 가장 먼저 준비해야 할 것 3가지를 알려줘."
#     )
#     print(result)

# await test_basic_invoke()

async def testBasicInvoke():
    result = await kernel.invoke_prompt(
        "서울 마포구에서 일반음식점을 창업할 때 가장 먼저 준비해야 할 것 3가지를 알려줘."
    )
    print(result)

await testBasicInvoke()

서울 마포구에서 일반음식점을 창업할 때 가장 먼저 준비해야 할 세 가지는 다음과 같습니다:

1. **사업 계획서 작성**:
   - 음식점의 컨셉, 목표 고객, 메뉴 구성, 가격대 등을 명확히 정의한 사업 계획서를 작성하세요. 이 계획서는 창업 과정 전반에 걸쳐 가이드 역할을 하며, 금융기관에서 대출을 받거나 투자자를 구할 때도 필요합니다.

2. **입지 선정 및 임대 계약**:
   - 음식점의 성공에 있어 좋은 위치 선정은 매우 중요합니다. 목표 고객층에 적합한 유동 인구가 많은 곳을 탐색하고, 임대료와 계약 조건 등을 꼼꼼히 검토한 후 임대 계약을 체결하세요.

3. **관련 인허가 및 등록 절차**:
   - 음식점 영업을 위해서는 식품위생법에 따른 영업허가를 받아야 합니다. 사업자등록증 발급, 건강진단서 준비 등 필요한 모든 서류를 구비하고 신청하세요. 또한, 위생교육도 필수적으로 이수해야 합니다.

이 외에도 인테리어, 장비 구입, 인력 채용 등의 준비가 필요하며, 각 단계에서 철저한 계획과 실행이 중요합니다.


---
## 3. Plugin 만들기 — `@kernel_function`

> **핵심 개념**: `@kernel_function` 데코레이터로 Python 함수를 LLM이 호출할 수 있는 도구로 만듭니다.
>
> F&B 시스템에서는 이 패턴으로:
> - 국가법령정보센터 API 호출
> - 카카오 지도 API 호출  
> - 몬테카를로 시뮬레이션 실행
>
> 등을 구현하게 됩니다.

In [5]:
from typing import Annotated
from pkg_resources import Requirement
from semantic_kernel.functions import kernel_function

# ----------------------------------------
# [F&B 법령 플러그인 예시]
# 실제 구현에서는 국가법령정보센터 API를 호출합니다.
# 지금은 Mock 데이터로 개념만 익힙니다.
# ----------------------------------------
# class FnbLegalPlugin:
#     """F&B 창업 관련 법령·인허가 정보를 제공하는 플러그인"""

#     @kernel_function(
#         name="get_license_requirements",
#         description="F&B 업종과 규모에 따른 영업 인허가 요건과 필요 서류 목록을 반환합니다."
#     )
#     def get_license_requirements(
#         self,
#         # Annotated의 두 번째 인자가 LLM에게 전달되는 파라미터 설명입니다
#         business_type: Annotated[str, "업종 (예: 일반음식점, 카페, 분식집)"],
#         area_sqm: Annotated[float, "영업 면적 (제곱미터)"],
#     ) -> str:
#         """Mock: 실제 구현에서는 법령정보 API 호출"""
#         # 실제로는 requests.get("https://www.law.go.kr/...") 형태로 호출
#         requirements = {
#             "일반음식점": [
#                 "식품위생법 제37조 영업신고 (관할 보건소)",
#                 "식품위생교육 이수 (6시간, 영업 전)",
#                 "위생등급 지정 신청 (선택)",
#             ],
#             "카페": [
#                 "식품위생법 제37조 영업신고 (휴게음식점)",
#                 "식품위생교육 이수 (3시간)",
#             ],
#         }
#         docs = requirements.get(business_type, ["해당 업종 정보 없음"])
#         area_note = "소방 완비 증명서 필요" if area_sqm >= 300 else ""
#         return "\n".join(docs) + ("\n" + area_note if area_note else "")

#     @kernel_function(
#         name="get_tax_type_guide",
#         description="예상 연매출을 기반으로 적합한 사업자 과세유형(간이/일반)을 안내합니다."
#     )
#     def get_tax_type_guide(
#         self,
#         annual_revenue_krw: Annotated[int, "예상 연 매출액 (원 단위)"],
#     ) -> str:
#         """Mock: 간이과세 vs 일반과세 기준 안내"""
#         # 2024년 기준 간이과세 기준: 연 매출 1억 400만원 미만
#         threshold = 104_000_000
#         if annual_revenue_krw < threshold:
#             return (
#                 "[간이과세자 해당]\n"
#                 "- 부가세: 업종별 부가가치율 적용 (일반과세 대비 세 부담 낮음)\n"
#                 "- 세금계산서 발행 불가 (법인 거래처 있으면 불리)\n"
#                 "- 기준: 연 매출 1억 400만원 미만"
#             )
#         else:
#             return (
#                 "[일반과세자 해당]\n"
#                 "- 부가세: 매출의 10% (매입세액 공제 가능)\n"
#                 "- 세금계산서 발행 가능\n"
#                 "- 기준: 연 매출 1억 400만원 이상"
#             )

# print("FnbLegalPlugin 정의 완료")


class FnbLegalPlugin:
    """F&B 창업 관련 법령,인허가 정보를 제공하는 플러그인"""

    @kernel_function(
        name="getLicenseRequirements",
        description="F&B 업종과 규모에 따른 영업 인허가 요건과 필요 서류 목록을 반환합니다.",
    )
    def getLicenseRequirements(
        self,
        # Anotated의 두 번째 인자가 LLM에게 전달되는 파라미터 설명입니다
        businessType: Annotated[str, "업종 (예: 일반음식점, 카페, 분식집)"],
        area_sqm: Annotated[float, "영업 면적 (제곱미터)"],
    ) -> str:
        """Mock: 실제 구현에서는 법령정보 api 호출"""
        # 실제로는 requests.get("https://www.law.go.kr/...") 형태로 호출
        requirements = {
            "일반음식점": [
                "식품위생법 제37조 영업신고 (관할 보건소)",
                "식품위생교육 이수 (6시간, 영업 전)",
                "위생등급 지정 신청 (선택)",
            ],
            "카페": [
                "식품위생법 제37조 영업신고 (휴게음식점)",
                "식품위생교육 이수 (3시간)",
            ],
        }
        docs = requirements.get(businessType, ["해당 업종 정보 없음"])
        areaNote = "소방 완비 증명서 필요" if area_sqm >= 300 else ""
        return "\n".join(docs) + ("\n" + areaNote if areaNote else "")

    @kernel_function(
        name="getTaxTypeGuide",
        description="예상 연매출을 기반으로 적합한 사업자 과세유형(간이/일반)을 안내합니다.",
    )
    def getTaxTypeGuide(
        self,
        annualRevenueKrw: Annotated[int, "예상 연 매출액 (원 단위)"],
    ) -> str:
        """Mock: 간이과세 vs 일반과세 기준 안내"""
        # 2024년 기준 간이과세 기준: 연 매출 1억 400만원 미만
        threshold = 104_000_000
        if annualRevenueKrw < threshold:
            return (
                "[일반과세자 해당]\n"
                "- 부가세: 업종별 부가가치율 적용 (일반과세 대비 세 부담 낮음)\n"
                "- 세금계간서 발행 불가 (법인 거래처 있으면 불리)\n"
                "- 기준: 연 매출 1억 400만원 미만"
            )
        else:
            return (
                "[일반과세자 해당]\n"
                "- 부가세: 매출의 10% (매입세액 공제 가능)\n"
                "- 세금계산서 발행 가능\n"
                "- 기준: 연 매출 1억 400만원 이상"
            )


print("FnbLegalPlugin 정의 완료")

FnbLegalPlugin 정의 완료


In [6]:
# # Kernel에 플러그인 등록
# kernel.add_plugin(FnbLegalPlugin(), plugin_name="FnbLegal")

# # 등록된 함수 목록 확인
# for name, func in kernel.plugins["FnbLegal"].functions.items():
#     print("[함수]", name, ":", func.description)

# Kernel에 플러그인 등록
kernel.add_plugin(FnbLegalPlugin(), plugin_name="FnbLegal")

# 등록된 함수 목록 확인
for name, func in kernel.plugins["FnbLegal"].functions.items():
    print("[함수]", name, ":", func.description)

[함수] getLicenseRequirements : F&B 업종과 규모에 따른 영업 인허가 요건과 필요 서류 목록을 반환합니다.
[함수] getTaxTypeGuide : 예상 연매출을 기반으로 적합한 사업자 과세유형(간이/일반)을 안내합니다.


---
## 4. 플러그인을 LLM이 자동 호출하게 만들기

> `FunctionChoiceBehavior.Auto()` — LLM이 대화 맥락에서 어떤 함수를 언제 호출할지 스스로 결정합니다.

In [7]:
from semantic_kernel.connectors.ai.open_ai import AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import (
    FunctionChoiceBehavior,
)
from semantic_kernel.contents import ChatHistory

# async def chat_with_legal_plugin(user_question: str):
#     """
#     LLM이 FnbLegalPlugin 함수를 자동으로 선택·호출하여 답변합니다.

#     내부 흐름:
#     1. 사용자 질문 → LLM 전달
#     2. LLM: 어떤 함수 호출할지 결정 → get_license_requirements 호출
#     3. Python 함수 실행 → 결과 LLM에 반환
#     4. LLM: 결과 기반으로 최종 답변 생성
#     """
#     history = ChatHistory()
#     history.add_system_message(
#         "당신은 F&B 창업 전문 어드바이저입니다. "
#         "제공된 도구를 활용하여 정확한 정보를 제공하세요."
#     )
#     history.add_user_message(user_question)
#     # Auto: LLM이 필요한 함수를 스스로 결정해서 호출
#     settings = AzureChatPromptExecutionSettings()
#     settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

#     chat_service = kernel.get_service(type=AzureChatCompletion)
#     result = await chat_service.get_chat_message_content(
#         chat_history=history,
#         settings=settings,
#         kernel=kernel,  # 함수 호출에 필요한 kernel 전달
#     )
#     return result.content


# # 테스트: LLM이 get_license_requirements 함수를 자동으로 호출해야 합니다
# answer = await chat_with_legal_plugin(
#     "서울에서 50평짜리 일반음식점을 열려고 해. 어떤 인허가가 필요해?"
# )
# print(answer)


async def chatWithLegalPlugin(userQuestion: str):
    """
    LLM이 FnbLegalPlugin 함수를 자동으로 선택·호출하여 답변합니다.

    내부 흐름:
    1. 사용자 질문 → LLM 전달
    2. LLM: 어떤 함수 호출할지 결정 → get_license_requirements 호출
    3. Python 함수 실행 → 결과 LLM에 반환
    4. LLM: 결과 기반으로 최종 답변 생성
    """
    history = ChatHistory()
    history.add_system_message(
        "당신은 F&B 창업 전문 어드바이저입니다. "
        "제공된 도구를 활용하여 정확한 정보를 제공하세요."
    )
    history.add_user_message(userQuestion)
    # Auto: LLM이 필요한 함수를 스스로 결정해서 호출
    settings = AzureChatPromptExecutionSettings()
    settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

    chatService = kernel.get_service(type=AzureChatCompletion)
    result = await chatService.get_chat_message_content(
        chat_history=history,
        settings=settings,
        kernel=kernel,  # 함수 호출에 필요한 kernel 전달
    )
    return result.content


# 테스트: LLM이 getLicenseRequirements 함수를 자동으로 호출해야 합니다
answer = await chatWithLegalPlugin(
    "서울에서 50평짜리 일반음식점을 열려고 해. 어떤 인허가가 필요해?"
)
print(answer)

서울에서 50평(165제곱미터) 일반음식점을 열기 위해 필요한 인허가는 다음과 같습니다:

1. **식품위생법 제37조 영업신고**: 관할 보건소에 신고해야 합니다.
2. **식품위생교육 이수**: 영업 전 6시간의 식품위생교육을 이수해야 합니다.
3. **위생등급 지정 신청**: 선택 사항으로, 위생등급 지정을 신청할 수 있습니다. 

각 요건은 해당 관할 구역의 보건소에서 진행되니, 구체적인 절차는 해당 구청 또는 보건소에 문의해 보시기 바랍니다.


In [8]:
# # 두 번째 테스트: 과세유형 함수 자동 호출 확인
# answer2 = await chat_with_legal_plugin(
#     "예상 연매출이 8천만원인데 간이과세자로 등록하는 게 맞을까?"
# )
# print(answer2)

# 두 번째 테스트: 과세유형 함수 자동 호출 확인
answer2 = await chatWithLegalPlugin(
    "예상 연매출이 8천만원인데 간이과세자로 등록하는게 맞을까?"
)
print(answer2)

예상 연매출이 8천만 원이라면 간이과세자로 등록하는 것이 일반적으로 적합합니다. 간이과세자는 연 매출 1억 4백만 원 미만인 경우에 해당하며, 부가세 측면에서 업종별 부가가치율이 적용되어 일반과세자와 비교했을 때 세 부담이 낮아질 수 있습니다. 다만, 간이과세자는 세금계산서 발행이 불가능하므로 법인 거래처와의 거래에는 불리할 수 있습니다.


---
## 5. 상권 분석 플러그인 추가 — 두 플러그인 동시 활용

In [9]:
import json

# class FnbLocationPlugin:
#     """상권·위치 분석 플러그인 (Mock — 실제 구현 시 카카오 지도 API 연동)"""

#     @kernel_function(
#         name="analyze_commercial_area",
#         description="특정 위치의 상권 정보(유동인구, 경쟁업체 수, 임대료 수준)를 분석합니다."
#     )
#     def analyze_commercial_area(
#         self,
#         location: Annotated[str, "분석할 위치 (예: 서울 마포구 홍대입구역 인근)"],
#         business_type: Annotated[str, "창업 업종"],
#     ) -> str:
#         """Mock: 실제 구현에서는 카카오 Local API + 소상공인시장진흥공단 API 호출"""
#         # 실제: requests.get("https://dapi.kakao.com/v2/local/search/keyword.json", ...)
#         mock_data = {
#             "location": location,
#             "business_type": business_type,
#             "daily_floating_population": 45000,
#             "competitor_count_500m": 12,
#             "avg_monthly_rent_krw_per_sqm": 85000,
#             "commercial_rating": "A",
#             "peak_hours": ["12:00-14:00", "18:00-21:00"],
#         }
#         return json.dumps(mock_data, ensure_ascii=False, indent=2)
# # Kernel에 두 번째 플러그인 추가
# kernel.add_plugin(FnbLocationPlugin(), plugin_name="FnbLocation")
# print("FnbLocationPlugin 등록 완료")


class FnbLocationPlugin:
    """상권·위치 분석 플러그인 (Mock — 실제 구현 시 카카오 지도 API 연동)"""

    @kernel_function(
        name="analyzeCommercialArea",
        description="특정 위치의 상권 정보(유동인구, 경쟁업체 수, 임대료 수준)를 분석합니다.",
    )
    def analyzeCommercialArea(
        self,
        location: Annotated[str, "분석할 위치 (예: 서울 마포구 홍대입구역 인근)"],
        businessType: Annotated[str, "창업 업종"],
    ) -> str:
        """Mock: 실제 구현에서는 카카오 Local API + 소상공인시장진흥공단 API 호출"""
        # 실제: requests.get("https://dapi.kakao.com/v2/local/search/keyword.json", ...)
        mock_data = {
            "location": location,
            "business_type": businessType,
            "daily_floating_population": 45000,
            "competitor_count_500m": 12,
            "avg_monthly_rent_krw_per_sqm": 85000,
            "commercial_rating": "A",
            "peak_hours": ["12:00-14:00", "18:00-21:00"],
        }
        return json.dumps(mock_data, ensure_ascii=False, indent=2)


# Kernel에 두 번째 플러그인 추가
kernel.add_plugin(FnbLocationPlugin(), plugin_name="FnbLocation")
print("FnbLocationPlugin 등록 완료")

FnbLocationPlugin 등록 완료


In [10]:
# 복합 질문: LLM이 두 플러그인 중 적절한 것을 선택해서 호출
answer3 = await chatWithLegalPlugin(
    "홍대 근처에서 카페 창업을 준비 중인데, 상권 분석이랑 필요한 인허가를 함께 알려줘."
)
print(answer3)

홍대 근처에서 카페 창업을 고려할 때 유용한 정보는 다음과 같습니다:

### 상권 분석
- **일일 유동인구**: 약 45,000명
- **경쟁업체 수 (반경 500m 내)**: 12곳
- **평균 월 임대료 수준**: 평당 약 85,000원
- **상권 평가**: A 등급
- **피크 시간대**: 12:00-14:00, 18:00-21:00

### 인허가 요건 및 필요 서류
1. **식품위생법 제37조 영업신고 (휴게음식점)**
2. **식품위생교육 이수 (3시간)**

이를 통해 적절한 입지와 필요한 인허가 절차를 준비하실 수 있습니다. 추가적으로 예상 연매출을 기반으로 한 세금 관련 안내가 필요하시면 말씀해 주세요.


---
## 6. ✅ STEP 01 정리

| 배운 것 | 코드 | 다음 단계 연결 |
|---------|------|---------------|
| `Kernel` 생성 + AI 서비스 등록 | `Kernel()` + `add_service()` | 모든 에이전트의 기반 |
| `@kernel_function` 플러그인 | `FnbLegalPlugin`, `FnbLocationPlugin` | STEP 02에서 에이전트에 붙임 |
| 함수 자동 호출 | `FunctionChoiceBehavior.Auto()` | STEP 02 ChatCompletionAgent 핵심 |
| 멀티 플러그인 동시 사용 | 두 플러그인 등록 후 복합 질의 | STEP 03 멀티 에이전트 기반 |

---

### 다음 STEP 미리보기
```
STEP 02: ChatCompletionAgent
  - Plugin을 장착한 전문 에이전트 (LegalTaxAgent, LocationAgent, FinanceAgent)
  - 에이전트 단독으로 대화 처리
  - ChatHistory로 멀티턴 대화 유지
```